# 🎬 Notebook 10 — Demo unificada (router automático Food-101 ↔ FoodSeg103-v2)

Este notebook integra los dos pipelines del proyecto en una sola interfaz, con un **router heurístico** que decide cuál usar según la escena de entrada:

- **Pipeline Food-101** (nb05/07): cuando la imagen parece **un plato compuesto** (mono-plato).
- **Pipeline FoodSeg103 v2** (nb09): cuando la imagen parece tener **varios ingredientes separados** (multi-ingrediente). Usa la versión mejorada: **SAM auto-mask + CLIP ensemble + filtro de confianza**.

La heurística mira la salida de YOLOv8-COCO: si hay 0 ó 1 caja de comida → Food-101; si hay ≥2 cajas o fuerte señal de vajilla con espacio → FoodSeg103-v2.

## Estructura

1. Setup: ambos pipelines + funciones core de FoodSeg103-v2 redefinidas inline.
2. Router heurístico `route(image)`.
3. Demo cualitativa sobre 4 imágenes (2 mono-plato + 2 multi-ingrediente).
4. Tabla resumen del proyecto (métricas de cada notebook).

> Requiere que estén corridos los nb03 (`weights/model_v1.pt`) y nb09 (lookups + checkpoints SAM/CLIP).

## ⚙️ Parte 1 — Setup

Cargamos los **dos pipelines** en la misma sesión:
- `pipe_f101`: `FoodVisionPipeline` del módulo `src.pipeline` (carga EfficientNet-B0 fine-tuneado).
- Para FoodSeg103 reinstanciamos SAM (con `SamAutomaticMaskGenerator`) + CLIP (con prompt ensemble) + YOLO (solo para el router) y redefinimos las funciones core del nb09 — los notebooks son self-contained, sin importar funciones entre sí.

In [ ]:
import sys
sys.path.append('..')

# Forzar tqdm clásico (texto) en vez de widgets
import tqdm.std, tqdm.notebook, tqdm.auto
tqdm.notebook.tqdm = tqdm.std.tqdm
tqdm.auto.tqdm = tqdm.std.tqdm

import json
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torch
import torchvision
from datasets import load_dataset
from ultralytics import YOLO
from segment_anything import sam_model_registry, SamAutomaticMaskGenerator
import open_clip

from src.config import (
    DATA_DIR, WEIGHTS_DIR, DEVICE, SEED,
    FOODSEG103_CLASSES, FOODSEG103_NUTRITION_PATH,
)
from src.pipeline import FoodVisionPipeline

rng = np.random.default_rng(SEED)
print(f'Device: {DEVICE}')

# ── Pipeline Food-101 ───────────────────────────────────────────────────
pipe_f101 = FoodVisionPipeline(WEIGHTS_DIR / 'model_v1.pt')

# ── SAM (auto-mask generator) ──────────────────────────────────────────
SAM_CKPT = WEIGHTS_DIR / 'sam_vit_b_01ec64.pth'
sam = sam_model_registry['vit_b'](checkpoint=str(SAM_CKPT)).to(DEVICE)
sam_auto = SamAutomaticMaskGenerator(
    sam, points_per_side=24, pred_iou_thresh=0.86,
    stability_score_thresh=0.92, min_mask_region_area=500,
)
print(f'SAM vit_b (auto-mask) cargado en {DEVICE}')

# ── CLIP ───────────────────────────────────────────────────────────────
clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
    'ViT-B-32', pretrained='openai', force_quick_gelu=True,
)
clip_model = clip_model.to(DEVICE).eval()
clip_tokenizer = open_clip.get_tokenizer('ViT-B-32')
print('CLIP ViT-B/32 (openai) cargado')

# ── YOLO (solo para el router; no para producir prompts a SAM) ─────────
yolo = YOLO('yolov8n.pt')
print(f'YOLOv8n cargado ({len(yolo.names)} clases COCO, uso: router)')

# ── Embeddings textuales CLIP con ENSEMBLE de templates ────────────────
CLIP_TEMPLATES = [
    'a photo of {name}, a type of food.',
    'a close-up photo of {name}.',
    'a photo of {name} on a plate.',
    'an image of fresh {name}.',
    'a photo of cooked {name}.',
    '{name} ingredient.',
]
feats = []
with torch.no_grad():
    for cls in FOODSEG103_CLASSES:
        name = cls.strip()
        prompts = [t.format(name=name) for t in CLIP_TEMPLATES]
        tokens = clip_tokenizer(prompts).to(DEVICE)
        f = clip_model.encode_text(tokens)
        f = f / f.norm(dim=-1, keepdim=True)
        f = f.mean(dim=0); f = f / f.norm()
        feats.append(f)
text_features = torch.stack(feats)
print(f'Embeddings textuales (ensemble {len(CLIP_TEMPLATES)} templates): shape {tuple(text_features.shape)}')

# ── Lookup nutricional ─────────────────────────────────────────────────
nutrition_db = json.load(open(FOODSEG103_NUTRITION_PATH))
print(f'Lookup FoodSeg103: {len(nutrition_db)} ingredientes')

# ── Funciones core del nb09 v2 (inline) ────────────────────────────────
FOOD_COCO = {'pizza','donut','cake','sandwich','hot dog','apple','banana','orange','broccoli','carrot'}
SCALE_REF = {'bowl','cup','plate','dining table','fork','knife','spoon','wine glass','bottle'}
PLATE_PORTION_G = 250.0
MIN_CONF = 0.20
TOP_N_MASKS = 12

def get_ingredient_nutrition(name):
    return nutrition_db.get(name) or nutrition_db.get(name.strip())

def yolo_detections(image, conf=0.25):
    """Salida de YOLO partida en food/ref para el router. (NO se usa para prompts.)"""
    res = yolo(image, conf=conf, verbose=False)[0]
    W, H = image.size
    food, ref = [], []
    for box, cls_id in zip(res.boxes.xyxy.cpu().numpy(), res.boxes.cls.cpu().numpy()):
        x1, y1, x2, y2 = box.astype(int)
        name = res.names[int(cls_id)]
        if name in FOOD_COCO: food.append((x1, y1, x2, y2, name))
        elif name in SCALE_REF: ref.append((x1, y1, x2, y2, name))
    return food, ref

def segment_with_sam_auto(image, top_n=TOP_N_MASKS, min_area_frac=0.005):
    arr = np.array(image.convert('RGB'))
    H, W = arr.shape[:2]
    raw = sam_auto.generate(arr)
    masks = [m['segmentation'].astype(bool) for m in raw
             if m['area'] >= min_area_frac * H * W]
    masks.sort(key=lambda m: -int(m.sum()))
    return masks[:top_n]

def classify_with_clip(crop_pil, topk=3):
    with torch.no_grad():
        x = clip_preprocess(crop_pil.convert('RGB')).unsqueeze(0).to(DEVICE)
        feat = clip_model.encode_image(x)
        feat = feat / feat.norm(dim=-1, keepdim=True)
        sims = (100.0 * feat @ text_features.T).softmax(dim=-1)[0]
    top_vals, top_idx = sims.topk(topk)
    return [(FOODSEG103_CLASSES[i.item()], v.item()) for i, v in zip(top_idx, top_vals)]

def crop_from_mask(image, mask, pad=8):
    ys, xs = np.where(mask)
    if len(xs) == 0: return None
    H, W = mask.shape
    return image.crop((max(0, int(xs.min())-pad), max(0, int(ys.min())-pad),
                       min(W, int(xs.max())+pad), min(H, int(ys.max())+pad)))

def analyze_dish_foodseg103(image, min_conf=MIN_CONF):
    """Pipeline FoodSeg103 v2 (nb09): SAM auto-mask + CLIP ensemble + filtro de confianza."""
    image = image.convert('RGB')
    masks = segment_with_sam_auto(image)
    if not masks:
        return dict(detections=[], total_kcal=0.0)
    accepted, rejected = [], []
    for mask in masks:
        crop = crop_from_mask(image, mask)
        if crop is None or min(crop.size) < 10: continue
        topk = classify_with_clip(crop, topk=3)
        cls, conf = topk[0]
        entry = dict(mask=mask, ingredient=cls, confidence=conf, area_px=int(mask.sum()))
        if conf >= min_conf and get_ingredient_nutrition(cls) is not None:
            accepted.append(entry)
        else:
            entry['status'] = 'indeterminado'
            rejected.append(entry)
    accepted_area = sum(e['area_px'] for e in accepted) or 1
    dets, total_kcal = [], 0.0
    for e in accepted:
        portion = PLATE_PORTION_G * e['area_px'] / accepted_area
        nut = get_ingredient_nutrition(e['ingredient'])
        kcal = nut['calories'] * portion / 100
        dets.append(dict(mask=e['mask'], ingredient=e['ingredient'], confidence=e['confidence'],
                         portion_g=portion, kcal=kcal, status='ok'))
        total_kcal += kcal
    for e in rejected:
        dets.append(dict(mask=e['mask'], ingredient=e['ingredient'], confidence=e['confidence'],
                         portion_g=0.0, kcal=0.0, status='indeterminado'))
    return dict(detections=dets, total_kcal=total_kcal,
                n_accepted=len(accepted), n_rejected=len(rejected))

print('\nAmbos pipelines listos. Funciones disponibles:')
print('  pipe_f101.analyze(image)        -> dict Food-101')
print('  analyze_dish_foodseg103(image)  -> dict FoodSeg103 v2 (auto-mask + ensemble + filtro)')

## 🚦 Parte 2 — Router heurístico

La decisión se basa en la salida de YOLO sobre la imagen:

| Señal | Decisión |
|---|---|
| `len(food_boxes) >= 2` Y área combinada > 25% imagen | **FoodSeg103** — claramente varios alimentos separados |
| `len(food_boxes) >= 1` Y hay vajilla cubriendo > 30% | **FoodSeg103** — plato compuesto en bowl/dish (típico de FoodSeg103) |
| Otros casos | **Food-101** — escena mono-plato, el clasificador entrenado debería resolverlo mejor |

El umbral está pensado para ser *conservador*: en la duda, prefiere Food-101 (que es el pipeline más maduro y con mejor accuracy en su dominio).

In [ ]:
def route(image):
    """Decide qué pipeline aplicar según la salida de YOLO.

    Returns:
        ('food101' | 'foodseg103', explanation_str)
    """
    image = image.convert('RGB')
    W, H = image.size
    img_area = W * H
    food, ref = yolo_detections(image)

    food_area = sum((x2-x1)*(y2-y1) for x1,y1,x2,y2,_ in food) / img_area
    ref_area  = sum((x2-x1)*(y2-y1) for x1,y1,x2,y2,_ in ref)  / img_area

    if len(food) >= 2 and food_area > 0.25:
        return 'foodseg103', f'≥2 cajas de comida cubriendo {food_area:.0%} → multi-ingrediente'
    if len(food) >= 1 and ref_area > 0.30:
        return 'foodseg103', f'comida + vajilla grande ({ref_area:.0%}) → plato compuesto'
    if len(food) == 0 and len(ref) == 0:
        return 'food101', 'YOLO no detecta nada distintivo → Food-101 conservador'
    return 'food101', f'{len(food)} caja(s) de comida, {len(ref)} de vajilla → mono-plato'


def analyze(image):
    """Interfaz unificada: aplica el pipeline elegido por el router."""
    if isinstance(image, str):
        image = Image.open(image)
    route_decision, reason = route(image)
    if route_decision == 'food101':
        r = pipe_f101.analyze(image)
        return dict(
            mode='food101', reason=reason,
            label=r['top_prediction']['class'].replace('_', ' '),
            confidence=r['top_prediction']['confidence'],
            total_kcal=r['nutrition']['estimated_portion']['calories'] if r['nutrition'] else None,
            detail=r,
        )
    r = analyze_dish_foodseg103(image)
    return dict(
        mode='foodseg103', reason=reason,
        ingredients=[d['ingredient'].strip() for d in r['detections']],
        total_kcal=r['total_kcal'],
        detail=r,
    )

print('Funciones listas:')
print('  route(image) -> ("food101" | "foodseg103", reason)')
print('  analyze(image) -> dict con mode, label/ingredients, kcal, detail')

## 🖼️ Parte 3 — Demo cualitativa sobre 4 imágenes

Elegimos 4 imágenes que cubren los dos extremos:

- **(a) Pizza de Food-101**: mono-plato típico → router debe elegir Food-101.
- **(b) Sushi de Food-101**: mono-plato con bowl posible → la decisión es interesante.
- **(c) bife_con_papas.jpeg**: foto real multi-ingrediente que ya está en el repo.
- **(d) Imagen de FoodSeg103**: plato compuesto con varios alimentos.

Mostramos cada imagen con la decisión del router, la etiqueta principal y las kcal totales.

In [ ]:
from src.config import FOOD101_CLASSES

# ── Cargar imágenes de demo ────────────────────────────────────────────
food101_test = torchvision.datasets.Food101(str(DATA_DIR), split='test',
                                            transform=None, download=False)

def find_first(label_name):
    label_idx = FOOD101_CLASSES.index(label_name)
    for i in range(len(food101_test)):
        img, lbl = food101_test[i]
        if lbl == label_idx:
            return img, label_name
    return None, None

img_pizza, _ = find_first('pizza')
img_sushi, _ = find_first('sushi')
img_bife    = Image.open('bife_con_papas.jpeg')

foodseg = load_dataset('EduardoPacheco/FoodSeg103', split='validation')
# Tomamos la primera imagen con al menos 3 ingredientes distintos en la máscara
img_multi = None
for i in range(len(foodseg)):
    s = foodseg[i]
    gt = np.array(s['label'])
    if gt.ndim == 3: gt = gt[..., 0]
    n_ing = len([c for c in np.unique(gt) if c > 0])
    if n_ing >= 3:
        img_multi = s['image'].convert('RGB')
        break

demo = [
    ('(a) Food-101 — pizza', img_pizza),
    ('(b) Food-101 — sushi', img_sushi),
    ('(c) bife_con_papas.jpeg', img_bife),
    ('(d) FoodSeg103 — plato compuesto', img_multi),
]

fig, axes = plt.subplots(1, 4, figsize=(20, 6))
for ax, (title, img) in zip(axes, demo):
    r = analyze(img)
    ax.imshow(img); ax.axis('off')
    if r['mode'] == 'food101':
        body = (f"ROUTER → Food-101\n{r['reason']}\n\n"
                f"Plato: {r['label']}\nConfianza: {r['confidence']:.2f}\n"
                f"Total: {r['total_kcal']:.0f} kcal")
        color = 'tab:blue'
    else:
        ing_str = ', '.join(r['ingredients'][:6]) + ('...' if len(r['ingredients']) > 6 else '')
        body = (f"ROUTER → FoodSeg103\n{r['reason']}\n\n"
                f"{len(r['ingredients'])} ingredientes: {ing_str}\n"
                f"Total: {r['total_kcal']:.0f} kcal")
        color = 'tab:green'
    ax.set_title(title, fontsize=10, fontweight='bold')
    ax.text(0.02, -0.05, body, transform=ax.transAxes, fontsize=8,
            color='white', va='top',
            bbox=dict(facecolor=color, alpha=0.85, pad=6, edgecolor='none'))

plt.tight_layout()
plt.savefig('unified_demo.png', dpi=110, bbox_inches='tight')
plt.show()

## 📊 Parte 4 — Tabla resumen del proyecto

Métricas finales por etapa del ROADMAP. Los valores se toman de los notebooks anteriores; ver cada notebook para el detalle metodológico.

In [ ]:
import pandas as pd

resumen = pd.DataFrame([
    {'Etapa': '1 · Baseline (Food-101)',           'Notebook': 'nb02/05',
     'Dataset': 'Food-101 test',          'Detección': '—',
     'Clasificación': 'Top-1 ~85% / Top-5 ~96%',   'MAPE': '~11.5%'},
    {'Etapa': '2a · Cascada YOLO+EffNet v1',        'Notebook': 'nb06',
     'Dataset': 'Food-101 test',          'Detección': 'YOLOv8-COCO (suma ingenua)',
     'Clasificación': 'EfficientNet-B0 (Food-101)','MAPE': '~238%'},
    {'Etapa': '2b · Cascada corregida v2',          'Notebook': 'nb07',
     'Dataset': 'Food-101 test',          'Detección': 'YOLO + NMS + filtro vajilla',
     'Clasificación': 'EfficientNet-B0 (Food-101)','MAPE': '~32%'},
    {'Etapa': '2c · Detección eval (AP@50)',        'Notebook': 'nb07',
     'Dataset': 'FoodSeg103 val',         'Detección': 'AP@50 = 0.116',
     'Clasificación': '—',                          'MAPE': '—'},
    {'Etapa': '3a · FoodSeg103 v1 (SAM+YOLO+CLIP)', 'Notebook': 'nb08',
     'Dataset': 'FoodSeg103 val',         'Detección': 'SAM con prompts YOLO (mIoU 0.26)',
     'Clasificación': 'CLIP zero-shot single (top-1 ~36%)', 'MAPE': '~74% media / 51% mediana'},
    {'Etapa': '3b · FoodSeg103 v2 (auto+ensemble)', 'Notebook': 'nb09',
     'Dataset': 'FoodSeg103 val',         'Detección': 'SAM auto-mask (mejor mIoU esperable)',
     'Clasificación': 'CLIP ensemble + filtro conf', 'MAPE': 'mejora vs v1 (ver nb09)'},
    {'Etapa': '4 · Demo unificada (router)',         'Notebook': 'nb10',
     'Dataset': 'mixto',                  'Detección': 'router heurístico YOLO',
     'Clasificación': 'switch entre Food-101 y nb09', 'MAPE': 'según rama elegida'},
])

print(resumen.to_string(index=False))

print('\n' + '=' * 72)
print('CONCLUSIONES')
print('=' * 72)
print('1. El pipeline Food-101 (nb02-05) sigue siendo el mejor para imágenes')
print('   mono-plato del estilo Food-101 (MAPE ~11.5%).')
print('2. La cascada YOLO + EfficientNet (nb06) sin las correcciones del nb07')
print('   degrada el sistema (MAPE explota a 238%) por duplicación de cajas')
print('   y conteo de vajilla como comida.')
print('3. Las correcciones del nb07 (NMS + filtro vajilla + reparto por área)')
print('   recuperan el pipeline a un MAPE razonable (~32%), todavía peor que')
print('   el baseline en escenas mono-plato pero ya útil.')
print('4. Para escenas multi-ingrediente (FoodSeg103), Food-101 estructuralmente')
print('   no puede funcionar: el clasificador no conoce ingredientes. El nb08')
print('   prueba SAM + CLIP zero-shot; la primera versión queda corta porque')
print('   YOLO no provee prompts útiles para los 103 ingredientes.')
print('5. El nb09 ataca eso: SAM auto-mask elimina la dependencia de YOLO, el')
print('   CLIP ensemble levanta top-1 unos puntos, y el filtro de confianza')
print('   evita los disparos catastróficos ("lamb 366 kcal" sobre una zanahoria).')
print('6. El router de este nb10 cierra el círculo: usa la herramienta correcta')
print('   para cada tipo de imagen.')
print()
print('TRABAJO FUTURO (fuera del alcance del TP):')
print('  - Fine-tunear EfficientNet-B0 sobre crops derivados de FoodSeg103')
print('    para reemplazar CLIP zero-shot y mejorar top-1 accuracy.')
print('  - Fine-tunear YOLO sobre FoodSeg103 (vía boxes derivadas de máscaras)')
print('    para tener detección específica de comida.')
print('  - Estimación de porción real desde la vajilla (escala física),')
print('    en vez del presupuesto fijo de 250 g por plato.')
print('=' * 72)